# Trader Sentiment Analysis

Exploring how Hyperliquid traders behave during Fear vs. Greed market conditions. We are analyzing trade frequency, position sizing, Win/Loss, drawdown proxy, and overall PnL.

## 1. Project Setup & Data Loading

load the data from `fear_greed_index.csv` and `historical_data.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")

# Load Datasets
sentiment = pd.read_csv('data/fear_greed_index.csv')
trades = pd.read_csv('data/historical_data.csv')

print(f"Sentiment Data Shape: {sentiment.shape}")
print(f"Trade Data Shape: {trades.shape}")

display(sentiment.head(3))
display(trades.head(3))


## 2. Data Quality Checks

Checking for any missing data or duplicates to ensure the integrity of the analysis.

In [ ]:
print("--- Missing Values ---")
print("Sentiment:\n", sentiment.isnull().sum())
print("\nTrades:\n", trades.isnull().sum())

print("\n--- Duplicates ---")
print(f"Sentiment duplicates: {sentiment.duplicated().sum()}")
print(f"Trade duplicates: {trades.duplicated().sum()}")


## 3. Data Cleaning & Alignment

Standardizing column names, extracting dates, and fixing any timezone disparities.

In [ ]:
# Extract dates for alignment
sentiment['date'] = pd.to_datetime(sentiment['date']).dt.date

# Trades timestamps are formatted as dd-mm-yyyy HH:MM
trades['Timestamp IST'] = pd.to_datetime(trades['Timestamp IST'], format='%d-%m-%Y %H:%M', errors='coerce')
trades['date'] = trades['Timestamp IST'].dt.date

# Clean up column names 
trades.columns = [c.lower().replace(' ', '_') for c in trades.columns]

# Drop any unparseable dates
trades = trades.dropna(subset=['date'])

print(f"Date range in trades: {trades['date'].min()} to {trades['date'].max()}")
print(f"Date range in sentiment: {sentiment['date'].min()} to {sentiment['date'].max()}")


## 4. Feature Engineering & Merging

build key metrics: Win Rate, Long/Short ratios, Average Position Size, Drawdown proxies, and PnL. Then merge on the standardized `date` column.

In [ ]:
trades['is_win'] = trades['closed_pnl'] > 0
trades['is_long'] = trades['direction'].str.lower().str.contains('buy', na=False) | trades['direction'].str.lower().str.contains('long', na=False)

# Aggregate daily metrics per trader
daily_trader = trades.groupby(['date', 'account']).agg(
    daily_pnl=('closed_pnl', 'sum'),
    trade_count=('account', 'count'),
    win_count=('is_win', 'sum'),
    avg_position_size=('size_usd', 'mean'),
    long_trades=('is_long', 'sum')
).reset_index()

daily_trader['win_rate'] = daily_trader['win_count'] / daily_trader['trade_count']
daily_trader['short_trades'] = daily_trader['trade_count'] - daily_trader['long_trades']
# Smooth long/short ratio to avoid divide by zero
daily_trader['ls_ratio'] = daily_trader['long_trades'] / (daily_trader['short_trades'] + 1)

# proxy for daily drawdown depth
daily_trader['is_loss_day'] = daily_trader['daily_pnl'] < 0
daily_trader['drawdown_proxy'] = np.where(daily_trader['daily_pnl'] < 0, daily_trader['daily_pnl'].abs(), 0)

# Combine with sentiment labels
trader_sentiment = pd.merge(daily_trader, sentiment[['date', 'value', 'classification']], on='date', how='inner')

print(f"Merged Dataset Shape: {trader_sentiment.shape}")
display(trader_sentiment.head(3))


## 5. Sentiment Grouping

Categorizing market states broadly into Fear and Greed to see overarching behavioral trends.

In [ ]:
def map_sentiment(c):
    if 'Fear' in c: return 'Fear'
    if 'Greed' in c: return 'Greed'
    return 'Neutral'

trader_sentiment['sentiment_group'] = trader_sentiment['classification'].apply(map_sentiment)
trades_analysis = trader_sentiment[trader_sentiment['sentiment_group'] != 'Neutral']

grouped_stats = trades_analysis.groupby('sentiment_group').agg(
    avg_pnl=('daily_pnl', 'mean'),
    median_pnl=('daily_pnl', 'median'),
    avg_win_rate=('win_rate', 'mean'),
    avg_ls_ratio=('ls_ratio', 'mean'),
    avg_trades=('trade_count', 'mean'),
    avg_pos_size=('avg_position_size', 'mean')
).round(2).reset_index()

display(grouped_stats)


## 6. Trader Segmentation & Behavioral Shifts

Grouping users by how often they trade. checking if frequent traders handle market fear differently than occasional traders.

In [ ]:
# Total lifetime trades per account
trader_totals = trades_analysis.groupby('account')['trade_count'].sum().reset_index()
med_freq = trader_totals['trade_count'].median()

# Segment accounts into Frequent and Infrequent
trader_totals['segment'] = np.where(trader_totals['trade_count'] >= med_freq, 'Frequent', 'Infrequent')
segment_map = dict(zip(trader_totals['account'], trader_totals['segment']))
trades_analysis['segment'] = trades_analysis['account'].map(segment_map)

# How do different segments react?
segment_sentiment = trades_analysis.groupby(['segment', 'sentiment_group'])['daily_pnl'].mean().unstack().round(2)
display(segment_sentiment)


## 7. Visualization

Generating required charts for the report based on the new metrics.

In [ ]:
os.makedirs('outputs/charts', exist_ok=True)
sns.set_palette('pastel')

# 1. Long/Short Bias vs Sentiment
plt.figure(figsize=(7, 5))
sns.boxplot(data=trades_analysis, x='sentiment_group', y='ls_ratio', showfliers=False)
plt.title('Long/Short Ratio Bias vs Market Sentiment')
plt.ylabel('Long / Short Ratio')
plt.xlabel('Sentiment')
plt.savefig('outputs/charts/ls_ratio_sentiment.png')

# 2. Drawdown Proxy Segmented
plt.figure(figsize=(7, 5))
sns.barplot(data=trades_analysis, x='sentiment_group', y='drawdown_proxy', hue='segment')
plt.title('Daily Drawdown Magnitude vs Sentiment by Segment')
plt.ylabel('Avg Negative PnL')
plt.xlabel('Sentiment')
plt.savefig('outputs/charts/drawdown_sentiment.png')

# 3. Position Size vs Sentiment segmented by Trader Type
plt.figure(figsize=(8, 5))
sns.barplot(data=trades_analysis, x='sentiment_group', y='avg_position_size', hue='segment')
plt.title('Average Position Size: Fear vs Greed')
plt.ylabel('Position Size USD')
plt.xlabel('Sentiment')
plt.legend(title='Trader Class')
plt.savefig('outputs/charts/position_size_vs_sentiment.png')


## 8. Predictive Model (Bonus)

 predict whether tomorrow will be net profitable based on today's sentiment, sizing, and long and short biases?

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Roll up the day level
daily_features = trades_analysis.groupby('date').agg(
    sentiment_val=('value', 'first'),
    avg_trades=('trade_count', 'mean'),
    avg_pos_size=('avg_position_size', 'mean'),
    ls_ratio=('ls_ratio', 'mean'),
    total_pnl=('daily_pnl', 'sum')
).sort_values('date')

# Target: Is the NEXT day profitable?
daily_features['profit_tmrw'] = (daily_features['total_pnl'].shift(-1) > 0).astype(int)
daily_features = daily_features.dropna()

X = daily_features[['sentiment_val', 'avg_trades', 'avg_pos_size', 'ls_ratio']]
y = daily_features['profit_tmrw']

if len(X) > 10:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    
    preds = clf.predict(X_test)
    print("Model Accuracy predicting next-day profitability:", round(accuracy_score(y_test, preds), 2))
    print("\nClassification Report:\n", classification_report(y_test, preds))
else:
    print("Not enough days of overlapping data for model evaluation.")
